# 1. Problem Definition
-

# 2. Import and Data Overview
-

In [33]:
# Import packages

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

In [34]:
# Import data
df = pd.read_csv("C:\\Users\\ajdpe\\Desktop\\Ficheiros\\4. PYTHON\\car-price-regression\\data\\train.csv")
display(df.head())
print(df.columns)
print(df.info())
print(df.describe())

,ID,Price,Levy,Manufacturer,Model,Prod. year,Category,Leather interior,Fuel type,Engine volume,Mileage,Cylinders,Gear box type,Drive wheels,Doors,Wheel,Color,Airbags
0,45654403,13328,1399,LEXUS,RX 450,2010,Jeep,Yes,Hybrid,3.5,186005 km,6.0,Automatic,4x4,04-May,Left wheel,Silver,12
1,44731507,16621,1018,CHEVROLET,Equinox,2011,Jeep,No,Petrol,3,192000 km,6.0,Tiptronic,4x4,04-May,Left wheel,Black,8
2,45774419,8467,-,HONDA,FIT,2006,Hatchback,No,Petrol,1.3,200000 km,4.0,Variator,Front,04-May,Right-hand drive,Black,2
3,45769185,3607,862,FORD,Escape,2011,Jeep,Yes,Hybrid,2.5,168966 km,4.0,Automatic,4x4,04-May,Left wheel,White,0
4,45809263,11726,446,HONDA,FIT,2014,Hatchback,Yes,Petrol,1.3,91901 km,4.0,Automatic,Front,04-May,Left wheel,Silver,4


Index(['ID', 'Price', 'Levy', 'Manufacturer', 'Model', 'Prod. year',
       'Category', 'Leather interior', 'Fuel type', 'Engine volume', 'Mileage',
       'Cylinders', 'Gear box type', 'Drive wheels', 'Doors', 'Wheel', 'Color',
       'Airbags'],
      dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 19237 entries, 0 to 19236
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                19237 non-null  int64  
 1   Price             19237 non-null  int64  
 2   Levy              19237 non-null  str    
 3   Manufacturer      19237 non-null  str    
 4   Model             19237 non-null  str    
 5   Prod. year        19237 non-null  int64  
 6   Category          19237 non-null  str    
 7   Leather interior  19237 non-null  str    
 8   Fuel type         19237 non-null  str    
 9   Engine volume     19237 non-null  str    
 10  Mileage           19237 non-null  str    
 11  Cylinders         1

In [35]:
# Check for value for categorical features
for col in df.select_dtypes("object"):
    print(f"Column: {col}")
    print(df[col].value_counts())
    print("\n")

Column: Levy
Levy
-       5819
765      486
891      461
639      410
640      405
        ... 
2308       1
4860       1
1641       1
1045       1
1901       1
Name: count, Length: 559, dtype: int64


Column: Manufacturer
Manufacturer
HYUNDAI          3769
TOYOTA           3662
MERCEDES-BENZ    2076
FORD             1111
CHEVROLET        1069
                 ... 
LAMBORGHINI         1
PONTIAC             1
SATURN              1
ASTON MARTIN        1
GREATWALL           1
Name: count, Length: 65, dtype: int64


Column: Model
Model
Prius                       1083
Sonata                      1079
Camry                        938
Elantra                      922
E 350                        542
                            ... 
Every Landy NISSAN SEREN       1
CL 600                         1
E 230 124                      1
RX 450 F SPORT                 1
Prius C aqua                   1
Name: count, Length: 1590, dtype: int64


Column: Category
Category
Sedan          8736
Jeep       

C:\Users\ajdpe\AppData\Local\Temp\ipykernel_7388\1978544129.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes("object"):


In [36]:
# Light clean on the dataset

# Exclude most expensive car (OPEL, mistake)
# Check -> print(df.sort_values(by='Price', ascending=False).head(10))
df = df[df['Price'] != df['Price'].max()]

# Exclude cars below 2000 euros (probably mistakes)
# Check -> print(df.sort_values(by='Price', ascending=True).head(20))
df = df[df['Price'] >= 2000]

# Transform 'Mileage' to numeric and exclude extreme values that are mistakes
df['Mileage'] = df['Mileage'].str.replace(' km', '').astype(int)
# Check -> print(df.sort_values(by='Mileage', ascending=False).head(10))
quantile_99_mileage = df['Mileage'].quantile(0.99)
df = df[df['Mileage'] <= quantile_99_mileage]

# Check out many rows were eliminated
print(f"Number of rows after cleaning: {len(df)}")


Number of rows after cleaning: 15873


## Conclusions

- The original dataset contains 19,237 rows and 18 columns, with a mix of numerical and categorical features. After some light cleaning outliers and possible mistakes in the dataset, the number of rows is now 15,873.

### Missing Data
- The 'Levy' column contains missing or invalid values represented as '-', which will require cleaning.

### Data Types
- Some features are stored as strings but represent numerical values:
- 'Mileage' and 'Doors' should be converted to integer.
- 'Engine volume' should be converted to float.

### Data Quality Observations
- 
- 'Mileage' includes units ('km') and will require cleaning.
- 'Doors' contains inconsistent formatting (e.g. '02-Mar', '04-May'), requiring standardization.
- Some categorical features appear to have limited categories (e.g. 'Wheel', 'Leather Interior').

### Initial Observations
Observations for Feature Engineering:
- Might make sense to transform date of production, which is a year, into a 'years since productions', subtracting the date of production to the current year.
- 'Mileage' has 'km', which needs to be cleaned.
- 'Doors' has a 2 and 4 door incorrectely typed, which needs to be transformed.
- Categorical features with only two categories such as 'Wheel' and 'Leather Interior' might be transformed into binary.
- Might be interesting to create a new feature 'Turbo' for when the 'Engine Volume' feature has 'Turbo' in it, as well as create intervals for the 'Engine Volume'. It might be interesting as well as for 'Fuel Type' to reduce categories.

# 3. Data Cleaning